In [ ]:
from utils_nmodel import *
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

from sbi.neural_nets import posterior_nn
from sbi.neural_nets.embedding_nets import FCEmbedding, PermutationInvariantEmbedding
from sbi.inference import NPE
from pathlib import Path
import sys
import json
import time
import pandas as pd

In [ ]:
# ================ Setting =============== #
# prior for theta0
a0 = -5.0
b0 = 5.0
# prior for theta1-thetaM
a = 0.0
b = 1.0

M = 10
x_dim = 2


sigma = 0.1 
obs_size = 1000

In [ ]:
task_id = 1

In [ ]:
theta_dummy = torch.rand(100, M + 1)
x_dummy = torch.rand(100, obs_size, 2)

In [ ]:
config = {
    "batch_size": 200,
    "summary_dim": 128,
    "embed_hidden_size": 128,
    "embed_num_layers": 3,
}


single_trial_net = FCEmbedding(
    input_dim=x_dim,
    num_hiddens=int(config["embed_hidden_size"]),
    num_layers=int(config["embed_num_layers"]),
    output_dim=int(config["summary_dim"]),
)

embedding_net = PermutationInvariantEmbedding(
    single_trial_net,
    trial_net_output_dim=int(config["summary_dim"]),
    num_hiddens=int(config["embed_hidden_size"]),
    num_layers=int(config["embed_num_layers"]),
    output_dim=int(config["summary_dim"]),
)

density_estimator = posterior_nn("maf", embedding_net=embedding_net)
inference = NPE(density_estimator=density_estimator)

# Build the neural net once so the modules exist before loading state_dict.
# We use a tiny dummy batch only for initializing the neural networks
inference.append_simulations(theta_dummy, x_dummy).train(
    max_num_epochs=1,
    training_batch_size=2,
    validation_fraction=0.5,
    show_train_summary=False,
)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(20, 11))
axes = axes.ravel()

# ---- SW plot (first position) ----
ax = axes[0]

theta_pre = torch.tensor(pd.read_csv(f"res_SW1_precond/theta_pre_task{task_id}.csv").values, dtype = torch.float32).contiguous()

A = get_A(M)
grids = torch.arange(0, 1.01, 0.01)
psi_grids = get_psi(grids, M)

pred_ys = psi_grids @ torch.linalg.inv(A) @ theta_pre.T


lower_bound = torch.quantile(pred_ys, 0.025, dim=1)
median = torch.quantile(pred_ys, 0.5, dim=1)
upper_bound = torch.quantile(pred_ys, 0.975, dim=1)

ax.plot(grids, torch.tanh(4.0 * grids - 2.0), color='red', label="True f")
ax.plot(grids, median, color='black', label="Median")
ax.fill_between(grids, lower_bound, upper_bound, color='blue', alpha=0.1,
                label="2.5-97.5 percentile range")
ax.set_title(f"Localization result, simulation cost = {50000}", fontsize=20)

# ---- NPE plots (remaining four) ----
for i, ref_size in enumerate([250000, 100000, 50000, 25000]):
    ax = axes[i+1]

    task_id = 1
    ref_size = int(ref_size)

    weights_path = Path("NPE_embed_model_diffrefsize") / f"npe_net_weights_task{task_id}_ref{ref_size}.pth"
    state_dict = torch.load(weights_path, map_location=device)
    inference._neural_net.load_state_dict(state_dict)
    inference._neural_net.eval()

    posterior = inference.build_posterior(inference._neural_net)

    data_obs = pd.read_csv(f"data_obs/data_obs_task{task_id}.csv")
    data_obs = torch.tensor(data_obs.values, dtype=torch.float32).contiguous().reshape(1, obs_size, x_dim)

    num_samples = 10000
    theta_post = posterior.sample((num_samples,), x=data_obs, show_progress_bars=False)

    A = get_A(M)
    grids = torch.arange(0, 1.01, 0.01)
    psi_grids = get_psi(grids, M)

    pred_ys = psi_grids @ torch.linalg.inv(A) @ theta_post.T

    lower_bound = torch.quantile(pred_ys, 0.025, dim=1)
    median = torch.quantile(pred_ys, 0.5, dim=1)
    upper_bound = torch.quantile(pred_ys, 0.975, dim=1)

    ax.plot(grids, torch.tanh(4.0 * grids - 2.0), color='red', label="True f")
    ax.plot(grids, median, color='black', label="Median")
    ax.fill_between(grids, lower_bound, upper_bound, color='blue', alpha=0.1,
                    label="2.5-97.5 percentile range")

    ax.set_title(f"NPE, simulation cost = {ref_size}", fontsize=20)


y_lims = [ax.get_ylim() for ax in axes[:5]]
y_min = min(lim[0] for lim in y_lims)
y_max = max(lim[1] for lim in y_lims)

for ax in axes[:5]:
    ax.set_ylim(y_min, y_max)
    ax.tick_params(axis="both", labelsize=14)

# ---- put legend in the last empty subplot ----
handles, labels = axes[0].get_legend_handles_labels()

axes[5].axis("off")
axes[5].legend(
    handles,
    labels,
    loc="center",
    frameon=False,
    fontsize=20
)


plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig("comparison_seq_vs_sw1.pdf", dpi=300, bbox_inches="tight")
plt.show()